In [ ]:
import json
import shutil
import warnings
from pathlib import Path

from statsmodels.tools.sm_exceptions import InterpolationWarning
warnings.simplefilter('ignore', InterpolationWarning)

import config
from input.input import load_raw_data
from model import generics, hybrid_system_exp, grid_search_exp
from model.feature_selection import TimeSeriesFeatureSelector
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import Pipeline
from utils.compare_fs_vs_baseline import build_comparison
from utils.export_metrics_to_csv import save_csv

%load_ext autoreload
%autoreload 2

In [ ]:
# === Notebook de FS na janela de 10% (pct10) -- ARIMA-MLP hibrido (Additive) / lasso ===
# Mesma estrutura dos chamados_v4_fs_* ('auto'), mas: (a) lag_size_override =
# resolve_lag_size_pct(N - test_size, 0.10) por serie; (b) comparacao par-a-par
# SEMPRE contra o baseline pct10 DA PROPRIA FAMILIA (celula final), NUNCA
# contra o baseline 'auto'. experiment_id/model_name distintos, sem underscore.
# seed=42 (familia MLP). force=False. Unica acao: Restart Kernel -> Run All.
model = Pipeline([
    ('selector', TimeSeriesFeatureSelector(strategy='lasso')),
    ('estimator', MLPRegressor(activation='logistic', solver='lbfgs')),
])

series_list = ['airlines.txt', 'austres.txt', 'coloradoRiver.txt', 'sunspot.txt', 'windspeedfortaleza.txt', 'samurec.txt']

experiment_id = 'chamados_pct10_fs_lasso'
model_name = 'amv1pct10lasso'   # -> 1amv1pct10lasso.pkl (sem underscore, RUNBOOK.md 7)
normalize = True
force = False
model_exec = 10

experiment_params = {
    'linear_model_name': '1arima',
    'diff_kpss': False,
    'horizon': 1,
}

model_parameters = {
    'estimator__hidden_layer_sizes': [10, 20, 50],
    'estimator__max_iter': [1000],
}

# Comparacao par-a-par: baseline pct10 da PROPRIA familia (Parte 1). Nunca 'auto'.
baseline_experiment_id = 'chamados_pct10'
baseline_model_name = '1amv1pct10'
linear_model_name_to_exclude = '1arima'

experiment_dir = Path(config.MODEL_DATA_PATH) / experiment_id
experiment_dir_results = Path(config.ROOT_PATH) / 'results' / experiment_id

In [ ]:
# Sanity-check (mesmo padrao dos notebooks 'auto'): Pipeline.get_params(deep=True)
# expoe as chaves que GridSearch vai usar; e strategy <-> experiment_id/model_name
# consistentes entre si.
params = model.get_params(deep=True)
required_keys = {'selector__strategy', 'estimator__hidden_layer_sizes', 'estimator__max_iter'}
missing = required_keys - params.keys()
assert not missing, f'get_params(deep=True) nao expos: {missing}'

strategy_slug = model.named_steps['selector'].strategy.replace('_', '')
assert strategy_slug in experiment_id, f'{strategy_slug!r} nao em experiment_id={experiment_id!r}'
assert strategy_slug in model_name, f'{strategy_slug!r} nao em model_name={model_name!r}'
print(f'OK -- strategy={model.named_steps["selector"].strategy!r} consistente; keys expostas.')

In [ ]:
# Additive precisa do ARIMA pre-treinado sob o MESMO experiment_id. O ARIMA
# NAO depende de lag_size (auto_arima ajusta sobre a serie original, nao a
# janela) -- copia byte-a-byte de chamados/, o mesmo .pkl usado pela matriz
# 'auto'. Idempotente.
experiment_dir.mkdir(parents=True, exist_ok=True)
for base_name in series_list:
    serie = base_name.split('.')[0]
    src = Path(config.MODEL_DATA_PATH) / 'chamados' / f'{serie}_1arima.pkl'
    dst = experiment_dir / f'{serie}_1arima.pkl'
    assert src.exists(), f'ARIMA pre-treinado ausente: {src} -- rode arima_exec.ipynb antes.'
    shutil.copy(src, dst)
    print(f'{serie}: {src.name} -> {dst}')

In [ ]:
# Janela de 10%: lag_size_override = resolve_lag_size_pct(N - test_size, 0.10)
# por serie -- MESMA base que get_max_lag_to_consider (PACF sobre
# ts_univariate[0:-test_size]). Computado aqui, nada a editar.
# force=False: execution() (correcao de 2026-09-02) pula .pkl ja existente
# e nao-vazio -- re-Run All e idempotente.
lag_pct_por_serie = {}
for base_name in series_list:
    n_raw = len(load_raw_data(base_name))
    n_train = n_raw - int(config.TEST_SIZE * n_raw)
    lag_pct = grid_search_exp.resolve_lag_size_pct(n_train, pct=0.10)
    lag_pct_por_serie[base_name] = lag_pct
    print(f'{base_name}  N={n_raw}  N-test={n_train}  lag_pct={lag_pct}')
    exec_gs = grid_search_exp.GridSearch(
        hybrid_system_exp.Additive,
        model,
        model_parameters,
        experiment_id,
        base_name,
        model_name,
        force,
        normalize,
        experiment_params,
        model_exec=model_exec,
        use_val_slipt_for_prev=True,
        lag_size_override=lag_pct,
        estimator_random_state_base=grid_search_exp.MLP_RANDOM_STATE_BASE,  # CLAUDE.md 3.4 -- seed fixa das familias MLP
    )
    exec_gs.execution()

In [ ]:
from utils.export_metrics_to_csv import run_export_metrics_to_csv

df_metrics = run_export_metrics_to_csv(
    experiment_dir, experiment_dir_results / 'metrics.csv', detail=True,
)
df_metrics

In [ ]:
from utils.export_selected_features import run_export_selected_features

df_features = run_export_selected_features(
    experiment_dir, experiment_dir_results / 'selected_features.csv', detail=True,
)
df_features

In [ ]:
# Comparacao PAR-A-PAR: FS pct10 x baseline pct10 da MESMA familia.
# NUNCA contra o baseline 'auto' -- isola o efeito da selecao de features
# do efeito da definicao de janela. A chave do dict e so o rotulo da coluna
# de saida (o slug da estrategia).
fs_dirs = {experiment_id.rsplit('_', 1)[-1]: experiment_dir}
df_cmp = build_comparison(
    Path(config.MODEL_DATA_PATH) / baseline_experiment_id,
    fs_dirs,
    baseline_model_name=baseline_model_name,
    linear_model_name_to_exclude=linear_model_name_to_exclude,
)
save_csv(df_cmp, experiment_dir_results / 'comparison.csv',
         label='comparacao FS pct10 x baseline pct10 (par-a-par)')
df_cmp

In [ ]:
import json

metadata = {
    'experiment_id': experiment_id,
    'notebook': 'residual_hydridsystem/arima_mlp_pct10_lasso.ipynb',
    'tipo': 'fs pct10',
    'familia': 'ARIMA-MLP hibrido (Additive) / lasso',
    'janela': 'pct10 -- resolve_lag_size_pct(N - int(config.TEST_SIZE*N), 0.10)',
    'lag_pct_por_serie': {s: lag_pct_por_serie[s] for s in series_list},
    'series': series_list,
    'model_exec': model_exec,
    'seed': grid_search_exp.MLP_RANDOM_STATE_BASE,
    'model_parameters': model_parameters,
    'diff_kpss': experiment_params['diff_kpss'],
    'baseline_pareado': baseline_model_name,
    'strategy': 'lasso',
}
experiment_dir_results.mkdir(parents=True, exist_ok=True)
(experiment_dir_results / 'metadata.json').write_text(
    json.dumps(metadata, indent=2, default=str), encoding='utf-8'
)
print('metadata.json ->', experiment_dir_results / 'metadata.json')